# Coverless steganography baseline

Before running: turn on **GPU T4 x2** and **Internet**, and add the dataset. See `MANUAL.md`, section 4.

Change only the settings in the next cell.

In [ ]:
# ===== SETTINGS: change only these lines =====
REPO = "i-git-abhishek/Enhanced-Coverless-Steganography"
BRANCH = "rewrite-baseline"
CONFIG = "configs/voc2007.yaml"
RUN_EXTRACTION = True         # False if a finished cache dataset is attached
RUN_BASELINE = True           # one run with the n and M in CONFIG
SWEEP_N = []                  # for example [3, 4, 5, 6]
SWEEP_M = []                  # for example [3, 4, 5, 6]


In [ ]:
import os, subprocess

def run(command):
    print("\n$ " + command, flush=True)
    subprocess.run(command, shell=True, check=True)

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # no secret is needed for a public repo

folder = "/kaggle/working/" + REPO.split("/")[1]
if not os.path.isdir(folder):
    auth = f"x-access-token:{token}@" if token else ""
    # check=True is not used here so that a failed command cannot print the token.
    result = subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, f"https://{auth}github.com/{REPO}.git", folder], capture_output=True)
    if result.returncode != 0:
        raise SystemExit("Clone failed. Check REPO and BRANCH. For a private repo, add the GITHUB_TOKEN secret (MANUAL.md section 4).")
    subprocess.run(["git", "-C", folder, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
os.chdir(folder)
run("git log -1 --oneline")
subprocess.run("nvidia-smi", shell=True)  # absent when the GPU is off


In [ ]:
# Stop early if the code or data has a problem.
run("python -u run.py smoke")
run(f"python -u run.py check-data --config {CONFIG}")


In [ ]:
# Detector and ResNet-101 pass. A finished cache in an attached dataset is reused automatically.
if RUN_EXTRACTION:
    run(f"python -u run.py extract --config {CONFIG} --limit 4")
    run(f"python -u run.py extract --config {CONFIG}")


In [ ]:
# The stored detections must match a fresh detector pass. A mismatch does not stop the notebook,
# but must be read and reported (MANUAL.md section 11).
if RUN_EXTRACTION:
    subprocess.run(f"python -u run.py verify-detections --config {CONFIG} --sample 100", shell=True)


In [ ]:
# A failed setting must not lose finished runs: the next cell still creates the bundle.
failures = []

def attempt(command):
    print("\n$ " + command, flush=True)
    if subprocess.run(command, shell=True).returncode != 0:
        failures.append(command)

if RUN_BASELINE:
    attempt(f"python -u run.py baseline --config {CONFIG}")
if SWEEP_N:
    attempt(f"python -u run.py sweep --config {CONFIG} --param n --values " + " ".join(map(str, SWEEP_N)))
if SWEEP_M:
    attempt(f"python -u run.py sweep --config {CONFIG} --param M --values " + " ".join(map(str, SWEEP_M)))


In [ ]:
# Creates the one file to download from the Output tab.
run("python -u run.py bundle --output /kaggle/working/results_bundle.tar.gz")
print(open("experiments/RESULTS_DASHBOARD.md", encoding="utf-8").read()[:6000])
if failures:
    raise SystemExit("These commands failed. Finished runs are still in the bundle:\n" + "\n".join(failures))
